In [1]:
from Src.Data_utils import differencing
from dotenv import load_dotenv
from sqlalchemy import create_engine
import os
import pandas as pd

load_dotenv()
##Connect to sql database
engine = create_engine(
    f"mysql+pymysql://{os.getenv('MYSQL_USER')}:{os.getenv('MYSQL_PASSWORD')}@"
    f"{os.getenv('MYSQL_HOST')}:{os.getenv('MYSQL_PORT')}/{os.getenv('MYSQL_DB')}"
)

In [2]:
def feature_engineering():
    """
    USE: Puts I(1) and I(0) variables in different tables and creates correcly rescaled GDP per capita variable.

    Returns
    --------
    Data: pd.DataFrame
        2 dataframes one with the relevant I(1) and the other with relevat I(0) variables.
    """
    data = pd.read_sql("SELECT * FROM GDP_inference_clean", engine)

    data.columns = ["date", "Labor Productivity", "Unemployment Rate", "Federal Funds Rate", "Inflation", "GDP",
                    "Population", "Investment", "Government Spending","Consumption","Net Exports"] ##Renaming the columns
    data["GDP_per_capita"] = data["GDP"]*1000000 / data["Population"] ##Rescaling GDP so that GDP and population are both in thousands

    exclude = ["date", "Unemployment Rate", "GDP", "Population"] ##Excluding I(0) and redundant variables
    data_i1 = data.loc[:, ~data.columns.isin(exclude)]

    re_order_cols = ['Government Spending','Labor Productivity',"Consumption", 'Investment', 'GDP_per_capita',"Federal Funds Rate", "Net Exports",'Inflation'] ##Reorder columns for choletsky decomposition
    data_i1_reorder = data_i1[re_order_cols]
    exogenous_variables = data[["Unemployment Rate"]]##Selecing exogenous variables for VECM
    return data_i1_reorder,exogenous_variables

feature_engineering()

(     Government Spending  Labor Productivity  Consumption  Investment  \
 0                177.285                 3.4        530.9     592.754   
 1                183.665                 4.0        544.0     615.427   
 2                186.229                 3.5        563.2     598.643   
 3                190.202                 3.0        571.6     605.155   
 4                188.097                 1.5        583.5     640.790   
 ..                   ...                 ...          ...         ...   
 226             7165.663                 2.6      19949.5    4392.176   
 227             7247.662                 1.9      20226.0    4315.564   
 228             7313.597                 1.2      20462.2    4547.947   
 229             7496.339                 1.5      20746.4    4382.819   
 230             7580.229                 1.9      21007.3    4383.186   
 
      GDP_per_capita  Federal Funds Rate  Net Exports  Inflation  
 0       4550.029020                4.50   